Step 1: Bootstrap & install instructions

In [ ]:
!pip install --quiet anthropic pydantic
!pip install --upgrade --quiet ipython
from google.colab import userdata, drive
drive.mount('/content/drive')

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")
print('Ready!')

Step 2: Run fixture on 10 alerts.

In [ ]:
from astra_swarm.pipeline import astra_swarm_triage
from astra_swarm.cassette import cassette
import json
from pathlib import Path

alerts = json.loads(
    Path("/content/astra-swarm/data/synthetic/10_alerts.json").read_text()
)

with cassette("week02_milestone"):
    results = [astra_swarm_triage(a) for a in alerts]

out = Path("/content/astra-swarm/data/synthetic/week02_triage_results.json")
out.write_text(json.dumps([r.model_dump() for r in results], indent=2, default=str))

Step 3: Metrics

In [ ]:
from collections import Counter

w1 = json.loads(Path("/content/astra-swarm/data/synthetic/week01_triage_results.json").read_text())
w2 = json.loads(Path("/content/astra-swarm/data/synthetic/week02_triage_results.json").read_text())

def _sev_dist(runs, key_path):
    counts = Counter()
    for r in runs:
        cur = r
        for k in key_path:
            cur = cur[k]
        counts[cur] += 1
    return counts

print("Severity distribution:")
print(f"  Week 1: {dict(_sev_dist(w1, ['verdict', 'severity']))}")
print(f"  Week 2: {dict(_sev_dist(w2, ['investigation', 'severity']))}")

rounds = [r["investigation"]["rounds_used"] for r in w2]
print(f"\nRounds used (Week 2): min={min(rounds)}, max={max(rounds)}, avg={sum(rounds)/len(rounds):.1f}")

identity_pop = sum(1 for r in w2 if r["investigation"].get("identity_signals"))
print(f"Identity signals populated: {identity_pop}/{len(w2)}")